In [0]:
SELECT
    year,

    ROUND(AVG(pm25), 2) AS mean_pm25,
    ROUND(AVG(pm10), 2) AS mean_pm10,
    ROUND(AVG(no2), 3) AS mean_no2,
    ROUND(AVG(ozone), 3) AS mean_ozone,

    ROUND(AVG(rainfall), 4) AS mean_rainfall,
    ROUND(AVG(wind_speed), 2) AS mean_wind,
    ROUND(AVG(humidity), 1) AS mean_humidity,
    ROUND(AVG(temp_c), 1) AS mean_temp,

    COUNT(pm25) AS pm25_hours,

    COUNT(
        CASE
            WHEN pm25 IS NOT NULL
             AND wind_speed IS NOT NULL
            THEN 1
        END
    ) AS pm25_wind_hours,

    COUNT(
        CASE
            WHEN pm25 IS NOT NULL
             AND rainfall IS NOT NULL
            THEN 1
        END
    ) AS pm25_rain_hours

FROM workspace.aq_gold.hourly_wide

GROUP BY year
ORDER BY year;


SELECT
    year,

    COUNT(*) AS paired_hours,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN rainfall > 0 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS pct_hours_raining,

    ROUND(AVG(pm25), 2) AS mean_pm25

FROM workspace.aq_gold.hourly_wide

WHERE pm25 IS NOT NULL
  AND rainfall IS NOT NULL

GROUP BY year
ORDER BY year;


SELECT
    CASE
        WHEN rainfall > 0 THEN 'raining'
        ELSE 'dry'
    END AS condition,

    ROUND(AVG(pm25), 2) AS mean_pm25,
    ROUND(AVG(pm10), 2) AS mean_pm10,
    ROUND(AVG(no2), 3) AS mean_no2,
    ROUND(AVG(ozone), 3) AS mean_ozone,

    COUNT(*) AS paired_hours

FROM workspace.aq_gold.hourly_wide

WHERE pm25 IS NOT NULL
  AND rainfall IS NOT NULL

GROUP BY
    CASE
        WHEN rainfall > 0 THEN 'raining'
        ELSE 'dry'
    END

ORDER BY condition;


SELECT
    CASE
        WHEN wind_speed < 1 THEN '1_still'
        WHEN wind_speed < 3 THEN '2_light'
        WHEN wind_speed < 6 THEN '3_moderate'
        ELSE '4_strong'
    END AS wind_band,

    ROUND(AVG(pm25), 2) AS mean_pm25,
    ROUND(AVG(pm10), 2) AS mean_pm10,
    ROUND(AVG(no2), 3) AS mean_no2,
    ROUND(AVG(ozone), 3) AS mean_ozone,

    COUNT(*) AS paired_hours

FROM workspace.aq_gold.hourly_wide

WHERE pm25 IS NOT NULL
  AND wind_speed IS NOT NULL

GROUP BY
    CASE
        WHEN wind_speed < 1 THEN '1_still'
        WHEN wind_speed < 3 THEN '2_light'
        WHEN wind_speed < 6 THEN '3_moderate'
        ELSE '4_strong'
    END

ORDER BY wind_band;



SELECT
    ROUND(CORR(pm25, wind_speed), 3) AS pm25_vs_wind,
    ROUND(CORR(pm25, rainfall), 3) AS pm25_vs_rain,
    ROUND(CORR(pm25, humidity), 3) AS pm25_vs_humidity,
    ROUND(CORR(pm25, temp_c), 3) AS pm25_vs_temp,

    ROUND(CORR(pm10, wind_speed), 3) AS pm10_vs_wind,
    ROUND(CORR(no2, wind_speed), 3) AS no2_vs_wind,

    ROUND(CORR(ozone, temp_c), 3) AS ozone_vs_temp,
    ROUND(CORR(ozone, no2), 3) AS ozone_vs_no2

FROM workspace.aq_gold.hourly_wide;




WITH yearly AS (
    SELECT
        year,

        ROUND(AVG(pm25), 3) AS mean_pm25,
        ROUND(AVG(wind_speed), 3) AS mean_wind,

        ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN rainfall > 0 THEN 1
                    ELSE 0
                END
            )
            /
            COUNT(rainfall),
            2
        ) AS pct_hours_raining

    FROM workspace.aq_gold.hourly_wide

    WHERE pm25 IS NOT NULL

    GROUP BY year
)

SELECT *
FROM yearly
ORDER BY year;